## N-Coloring

From [Potassco user guide](https://github.com/potassco/guide/releases/download/v2.2.0/guide.pdf), sec. 6.1.

Define the program logic

In [ ]:
from aspish import function_, Solver, constraint, choose, VariableSequence


node = function_('node', ('label',))
edge = function_('edge', ('start', 'end'))
color = function_('color', ('node', 'color'))


def n_coloring(n_color: int) -> list:
    fresh = VariableSequence()
    X, Y, C = fresh(3)
    return [
        # choose one color from 1..n_color for each node
        choose(color(X, C), C.between(1, n_color), 1, 1) << node(X),
        # adjacent nodes must not have the same color
        constraint(edge(X, Y), color(X, C), color(Y, C))
    ]

Create problem instance and run solver

In [ ]:
node_list = list(range(1, 7))
adjacency = {
    1: [2, 3, 4],
    2: [4, 5, 6],
    3: [1, 4, 5],
    4: [1, 2],
    5: [3, 4, 6],
    6: [2, 3, 5],
}
n_color = 3

sol = Solver()
sol.add(*n_coloring(n_color))
sol.add(*[node(n) for n in node_list])
sol.add(*[edge(x, y) for x, ys in adjacency.items() for y in ys])
assert sol.solve()

Verify the solution

answer = sol.get(color)
node_color = {a.node: a.color for a in answer}
for i, js in adjacency.items():
    assert all(node_color[i] != node_color[j] for j in js)